# Notebook 06 — Positional encoding bakeoff

Paired with `theory/06_positional_encodings.md`. **Mode:** theory-first.

## QHMPC

**Question.** Structural properties of three positional encoding schemes plus an extrapolation bakeoff:
1. Sinusoidal APE inner product as a function of offset.
2. RoPE rotation-angle structure across frequencies.
3. Train length 64 → test length 256: which scheme extrapolates?
4. RoPE base ablation.
5. Attention-to-distance decay for trained models.

**Method.** Experiments 1–2: structural / visualization, no training. 3–5: small copy task with three 1-layer models.

**Prediction.** *Paste §6.7 predictions.*

**Confidence.** *[low/medium/high]*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)

## Experiment 1 — sinusoidal APE inner products

For $d = 32$ and positions $0, \dots, 255$, plot $\langle p_i, p_0 \rangle / d$ as a function of $i$. Prop 6.2 predicts this depends only on the offset.

In [ ]:
def sinusoidal_pe(n, d, base=10000.0):
    pos = np.arange(n)[:, None]
    k = np.arange(d // 2)
    omega = base ** (-2 * k / d)
    angles = pos * omega[None, :]
    pe = np.zeros((n, d))
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles)
    return pe

n, d = 256, 32
pe = sinusoidal_pe(n, d)
dots = pe @ pe[0] / d

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(dots)
axes[0].set_xlabel('position j'); axes[0].set_ylabel('<p_j, p_0> / d')
axes[0].set_title('Sinusoidal PE inner product vs offset')

# Verify offset-dependence: compute <p_i, p_j> / d for all (i, j); show it's a function of j - i only.
inner = pe @ pe.T / d
axes[1].imshow(inner, cmap='RdBu', vmin=-1, vmax=1)
axes[1].set_xlabel('j'); axes[1].set_ylabel('i')
axes[1].set_title('<p_i, p_j> / d  (Toeplitz = offset-dependent)')
plt.tight_layout(); plt.show()

**Sub-finding 1.** *Is the matrix Toeplitz (constant along diagonals)? At what rate does the inner product decay? Does it oscillate?*

## Experiment 2 — RoPE rotation angles

For $d_k = 64$: list the period (in sequence steps to a full rotation) of each frequency band.

In [ ]:
d_k = 64
for base in [1000, 10000, 100000]:
    thetas = base ** (-2 * np.arange(d_k // 2) / d_k)
    periods = 2 * np.pi / thetas
    print(f'base = {base}')
    print(f'  fastest period (k=0): {periods[0]:.2f} positions')
    print(f'  median period:         {periods[len(periods)//2]:.2f}')
    print(f'  slowest period (k=d_k/2-1): {periods[-1]:.2e} positions')
    print()

# Plot periods vs frequency index, for a few bases
fig, ax = plt.subplots(figsize=(7, 4))
for base in [1000, 10000, 100000]:
    thetas = base ** (-2 * np.arange(d_k // 2) / d_k)
    periods = 2 * np.pi / thetas
    ax.semilogy(periods, 'o-', label=f'base = {base}')
ax.set_xlabel('frequency band k'); ax.set_ylabel('period (positions)')
ax.set_title(f'RoPE periods across bands, d_k = {d_k}')
ax.legend(); plt.tight_layout(); plt.show()

**Sub-finding 2.** *At base = 10000, how many bands have period < 64 (the training length we'll use below)? Those are the bands that finish full rotations within training range; they encode short-range position.*

## Experiment 3 — bakeoff on a length-64 → 256 extrapolation task

Simple copy task: input is a sequence of integers, target is the same sequence shifted. Test whether each positional scheme can still solve it at 4× training length.

In [ ]:
def rope_rotate(x, pos, base=10000.0):
    """x: (B, n, d). pos: (n,) positions. Returns rotated x."""
    B, n, d = x.shape
    assert d % 2 == 0
    k = torch.arange(d // 2, dtype=x.dtype)
    theta = base ** (-2 * k / d)                        # (d/2,)
    angles = pos.unsqueeze(-1) * theta.unsqueeze(0)     # (n, d/2)
    cos = angles.cos(); sin = angles.sin()              # (n, d/2)
    x1 = x[..., 0::2]; x2 = x[..., 1::2]                # (B, n, d/2)
    out1 = x1 * cos - x2 * sin
    out2 = x1 * sin + x2 * cos
    out = torch.stack([out1, out2], dim=-1).flatten(-2)
    return out

class Attn(nn.Module):
    def __init__(self, d, pe_type, n_max=256, rope_base=10000.0):
        super().__init__()
        self.d, self.pe_type, self.rope_base = d, pe_type, rope_base
        self.W_Q = nn.Linear(d, d, bias=False)
        self.W_K = nn.Linear(d, d, bias=False)
        self.W_V = nn.Linear(d, d, bias=False)
        self.W_O = nn.Linear(d, d, bias=False)
        if pe_type == 'learned':
            self.pe = nn.Parameter(torch.randn(n_max, d) * 0.02)
        elif pe_type == 'sinusoidal':
            self.register_buffer('pe_buf', torch.tensor(sinusoidal_pe(n_max, d), dtype=torch.float32))

    def forward(self, x):
        B, n, d = x.shape
        if self.pe_type == 'learned':
            x = x + self.pe[:n]
        elif self.pe_type == 'sinusoidal':
            x = x + self.pe_buf[:n]
        Q = self.W_Q(x); K = self.W_K(x); V = self.W_V(x)
        if self.pe_type == 'rope':
            pos = torch.arange(n, dtype=Q.dtype, device=Q.device)
            Q = rope_rotate(Q, pos, base=self.rope_base)
            K = rope_rotate(K, pos, base=self.rope_base)
        scores = Q @ K.transpose(-1, -2) / (d ** 0.5)
        mask = torch.triu(torch.full((n, n), float('-inf'), device=x.device), diagonal=1)
        scores = scores + mask
        A = scores.softmax(-1)
        return self.W_O(A @ V)

# Copy task: input seq of tokens in {1..V}, target is the same shifted by one. Use 1-layer attention + output linear.
V = 32
d = 64

class Model(nn.Module):
    def __init__(self, pe_type, rope_base=10000.0):
        super().__init__()
        self.embed = nn.Embedding(V + 1, d)     # 0 is pad/start
        self.attn = Attn(d, pe_type, n_max=512, rope_base=rope_base)
        self.proj = nn.Linear(d, V + 1, bias=False)

    def forward(self, seq):
        x = self.embed(seq)
        y = self.attn(x)
        return self.proj(y)

def make_copy_batch(B, n):
    seq = torch.randint(1, V + 1, (B, n))
    inp = torch.cat([torch.zeros(B, 1, dtype=torch.long), seq[:, :-1]], dim=1)
    return inp, seq

def train(pe_type, rope_base=10000.0, n_train=64, steps=1500):
    torch.manual_seed(0)
    m = Model(pe_type, rope_base=rope_base)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    losses = []
    for _ in range(steps):
        inp, tgt = make_copy_batch(32, n_train)
        logits = m(inp)
        loss = F.cross_entropy(logits.reshape(-1, V + 1), tgt.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return m, losses

def eval_at_length(m, n_test, n_batches=20):
    m.eval()
    with torch.no_grad():
        accs = []
        for _ in range(n_batches):
            inp, tgt = make_copy_batch(32, n_test)
            try:
                logits = m(inp)
                acc = (logits.argmax(-1) == tgt).float().mean().item()
                accs.append(acc)
            except IndexError:
                return float('nan')    # learned PE out of range
    return np.mean(accs)

results = {}
for pe_type in ['learned', 'sinusoidal', 'rope']:
    m, losses = train(pe_type)
    acc_train = eval_at_length(m, 64)
    acc_test = eval_at_length(m, 256)
    results[pe_type] = (acc_train, acc_test, losses)
    print(f'{pe_type:>12}:  train(n=64) acc = {acc_train:.3f}   test(n=256) acc = {acc_test:.3f}')


**Sub-finding 3.** *Which scheme extrapolated? Did learned APE crash (NaN or index error)? Did sinusoidal degrade gracefully? Was RoPE stable?*

## Experiment 4 — RoPE base ablation

Does increasing the base help with extrapolation?

In [ ]:
print(f'{"base":>8}  {"acc(train=64)":>16}  {"acc(test=256)":>16}')
print('-' * 46)
for base in [1000, 10000, 100000, 1000000]:
    m, _ = train('rope', rope_base=base)
    acc_train = eval_at_length(m, 64)
    acc_test = eval_at_length(m, 256)
    print(f'{base:>8}  {acc_train:>16.3f}  {acc_test:>16.3f}')

**Sub-finding 4.** *Was there a base that helped extrapolation without hurting train accuracy? What does this say about the trade-off between short-range resolution (high-frequency bands) and long-range coverage (low-frequency bands)?*

## Finding

*3–6 sentences. Which scheme won at each length? Did the base ablation reveal the length-generalization trade-off? Next question: what do YaRN / Positional-Interpolation do to fix the degradation seen at long test lengths without retraining?*